# 06 | CatBoost SHAP interpretability and ensemble permutation analysis
**Purpose.** Explain the **fitted CatBoost member** of the final combined-feature ensemble. SHAP does not explain the hard-voting mechanism itself. An optional permutation experiment can evaluate the entire hard-voting predictor.

**Prerequisites:** notebooks 01, 03, and 04.  
**Outputs:** global/class-wise SHAP tables, figures, and optional ensemble permutation importance.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "config" / "experiment.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "config" / "experiment.json").exists():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks/ folder.")
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display
from src.gaia_pipeline import *
cfg = load_config(ROOT)
np.random.seed(cfg["random_state"])
print("Repository:", ROOT)
import joblib
import matplotlib.pyplot as plt
import shap
from sklearn.metrics import f1_score
balanced = read_object(ROOT, "preprocessed.joblib")["balanced"]
benchmark = read_object(ROOT, "benchmark.joblib")
ensemble = read_object(ROOT, "ensembles.joblib")
features = FEATURE_SETS["combined_features"]
classes = np.asarray(benchmark["classes"])
catboost = joblib.load(output_dir(ROOT) / "models" / "combined_features" / "CatBoost.joblib")
X_test = balanced.iloc[benchmark["test_idx"]][features]
y_test = benchmark["y_test"]
FIG = output_dir(ROOT) / "figures"; FIG.mkdir(exist_ok=True)

## Compute multiclass SHAP contributions
Use a fixed random sample of up to 3,000 held-out sources. Apply the **fitted** imputer from the original CatBoost pipeline; do not fit preprocessing again on the test data. Support SHAP's list or tensor multiclass formats.

The default uses CatBoost's native TreeSHAP calculation; set `shap_backend` to `tree_explainer` to use the original `shap.TreeExplainer` API where supported.

In [ ]:
X_raw = X_test.sample(n=min(len(X_test), cfg["shap_sample"]), random_state=cfg["random_state"])
X_values = pd.DataFrame(catboost[:-1].transform(X_raw), columns=features, index=X_raw.index)
if cfg.get("shap_backend", "catboost_native") == "tree_explainer":
    explainer = shap.TreeExplainer(catboost.named_steps["model"])
    shap_array = normalize_multiclass_shap(explainer.shap_values(X_values), len(classes))
else:
    from catboost import Pool
    native_values = catboost.named_steps["model"].get_feature_importance(Pool(X_values), type="ShapValues")
    shap_array = np.transpose(native_values[:, :, :-1], (0, 2, 1))
assert shap_array.shape == (len(X_values), len(features), len(classes))
print("SHAP tensor (sources, features, classes):", shap_array.shape)

## Global and class-specific mean absolute SHAP
Average the absolute contribution across sources and classes for global importance, and across sources for each individual spectral class. Save the full ranked tables to CSV.

In [ ]:
global_scores = np.abs(shap_array).mean(axis=(0, 2))
global_table = pd.DataFrame({"Feature": features, "Mean absolute SHAP": global_scores}).sort_values("Mean absolute SHAP", ascending=False)
save_table(global_table, ROOT, "shap_global.csv")
display(global_table)
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(global_table["Feature"][::-1], global_table["Mean absolute SHAP"][::-1])
ax.set(xlabel="Mean absolute SHAP", title="Global CatBoost SHAP importance")
fig.tight_layout(); fig.savefig(FIG / "shap_global.png"); plt.show()
classwise_table = pd.concat([pd.DataFrame({"Class": cls, "Feature": features, "Mean absolute SHAP": np.abs(shap_array[:, :, i]).mean(axis=0)}) for i, cls in enumerate(classes)], ignore_index=True)
classwise_table = classwise_table.sort_values(["Class", "Mean absolute SHAP"], ascending=[True, False])
save_table(classwise_table, ROOT, "shap_classwise.csv")
display(classwise_table.groupby("Class").head(5))

## Per-class SHAP summaries
Beeswarm summaries retain contribution direction and value information; bar plots summarize average magnitude. These attributions characterize the fitted Gaia-catalogue-label model, not an independent physical classifier.

In [ ]:
shap.summary_plot([shap_array[:, :, i] for i in range(len(classes))], X_values, feature_names=features, class_names=list(classes), plot_type="bar", show=False)
plt.title("Class-wise CatBoost SHAP importance"); plt.tight_layout()
plt.savefig(FIG / "shap_multiclass_bar.png", bbox_inches="tight"); plt.show()
for i, cls in enumerate(classes):
    shap.summary_plot(shap_array[:, :, i], X_values, feature_names=features, max_display=10, show=False)
    plt.title(f"CatBoost SHAP — class {cls}"); plt.tight_layout()
    plt.savefig(FIG / f"shap_beeswarm_{cls}.png", bbox_inches="tight"); plt.show()

## Optional permutation importance of the full hard-voting ensemble
Permutation importance measures **held-out predictive sensitivity**, not causal effect. Correlated variables can share information, so a low individual permutation score does not prove irrelevance.

In [ ]:
if cfg["run_permutation_importance"]:
    models = {name: joblib.load(output_dir(ROOT) / "models" / "combined_features" / f"{name}.joblib") for name in BASE_MODELS}
    X_perm = X_test.sample(n=min(1500, len(X_test)), random_state=cfg["random_state"])
    y_perm = y_test[X_test.index.get_indexer(X_perm.index)]
    def vote_predict(frame):
        pred = [np.asarray(models[name].predict(frame)).ravel().astype(int) for name in BASE_MODELS]
        prob = [models[name].predict_proba(frame) for name in BASE_MODELS]
        return hard_voting_from_arrays(pred, prob, len(classes))[0]
    baseline = f1_score(y_perm, vote_predict(X_perm), average="macro")
    rng = np.random.default_rng(cfg["random_state"])
    rows = []
    for feature in features:
        changes = []
        for _ in range(3):
            changed = X_perm.copy()
            changed[feature] = rng.permutation(changed[feature].to_numpy())
            changes.append(baseline - f1_score(y_perm, vote_predict(changed), average="macro"))
        rows.append({"Feature": feature, "Mean F1 decrease": np.mean(changes), "Std": np.std(changes)})
    permutation_table = pd.DataFrame(rows).sort_values("Mean F1 decrease", ascending=False)
    save_table(permutation_table, ROOT, "hard_voting_permutation_importance.csv")
    display(permutation_table.round(4))
else:
    print("Optional ensemble-level permutation importance is disabled.")